In [51]:
import sklearn
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)  # Disable line-wrapping
pd.set_option('display.max_rows', None)


coffea_caff = pd.read_csv(r"..\data\coll_caff_node_bin_w_class.csv")
coffee_data = coffea_caff.copy()

# coffea_caff_clim = pd.read_csv(r"..\data\coll_caff_node_clim_w_class.csv")
# coffee_data = coffea_caff_clim.copy()

#coffea_caff_env = pd.read_csv(r"..\data\coll_caff_node_env_w_class.csv")
#coffee_data = coffea_caff_env.copy()

#distinct_vals = coffee_data['caffeine_class'].value_counts()
#print(distinct_vals)

coffee_data.drop(columns=['specimen_id','specimen_name','Species_name', 'longitude','latitude', 'source_crs', 'mada_geom_point', 'sampled_layers', 'nodata_layers', 'is_categorical_encoded', 'clim_1_tmin1_jan_old', 'clim_13_tmax1_jan_old', 'clim_25_prec1_jan_old'],inplace=True)

coffee_data.head

<bound method NDFrame.head of      clim_1_tmin1_jan  clim_2_tmin2_feb  clim_3_tmin3_mar  clim_4_tmin4_apr  clim_5_tmin5_may  clim_6_tmin6_jun  clim_7_tmin7_jul  clim_8_tmin8_aug  clim_9_tmin9_sep  clim_10_tmin10_oct  ...  env_78_wat_17  env_78_wat_1.1  env_78_wat_18  env_78_wat_15  env_78_wat_19  env_78_wat_24  env_78_wat_21  env_78_wat_0  caffeine_percent  caffeine_class
0               195.5             195.0             186.0             169.0             124.0             108.0             108.0             119.0             133.0               173.0  ...              0               0              0              0              0              0              0             0             0.000               0
1               177.5             177.0             170.0             153.0             121.0             108.0             107.0             111.0             122.0               157.0  ...              0               0              0              0              0              

In [52]:
coffee_data.shape

(468, 160)

In [53]:
clim_columns = list(coffee_data.filter(like='clim_').columns) + [ 'env_71_alt', 'env_72_slo', 'env_73_asp', 'env_74_solrad', 'env_79_forcov' ]

clim_data = coffee_data[clim_columns]

env_data = coffee_data.drop(columns=clim_columns+['caffeine_percent', 'caffeine_class'])

# Optional: View the column names for confirmation
print("Clim Data Numerical Features:")
print(clim_data.columns)

print("\nEnv Data Categorical Features:")
print(env_data.columns)


Clim Data Numerical Features:
Index(['clim_1_tmin1_jan', 'clim_2_tmin2_feb', 'clim_3_tmin3_mar', 'clim_4_tmin4_apr', 'clim_5_tmin5_may', 'clim_6_tmin6_jun', 'clim_7_tmin7_jul', 'clim_8_tmin8_aug', 'clim_9_tmin9_sep', 'clim_10_tmin10_oct', 'clim_11_tmin11_nov', 'clim_12_tmin12_dec', 'clim_13_tmax1_jan', 'clim_14_tmax2_feb', 'clim_15_tmax3_mar', 'clim_16_tmax4_apr', 'clim_17_tmax5_may', 'clim_18_tmax6_jun', 'clim_19_tmax7_jul', 'clim_20_tmax8_aug', 'clim_21_tmax9_sep', 'clim_22_tmax10_oct', 'clim_23_tmax11_nov', 'clim_24_tmax12_dec', 'clim_25_prec1_jan', 'clim_26_prec2_feb', 'clim_27_prec3_mar', 'clim_28_prec4_apr', 'clim_29_prec5_may', 'clim_30_prec6_jun', 'clim_31_prec7_jul', 'clim_32_prec8_aug', 'clim_33_prec9_sep', 'clim_34_prec10_oct', 'clim_35_prec11_nov', 'clim_36_prec12_dec', 'clim_37_ann_mean_temp', 'clim_38_mean_diurn_range', 'clim_39_isotherm', 'clim_40_temp_season', 'clim_41_tmax_warmest_m', 'clim_42_tmin_coldest_m', 'clim_43_tannual_range', 'clim_44_tmean_wettest_q', 'clim_4

Managing NaN values

In [54]:
missing_data = coffee_data.isnull().sum()
missing_data

clim_1_tmin1_jan            18
clim_2_tmin2_feb            18
clim_3_tmin3_mar            18
clim_4_tmin4_apr            18
clim_5_tmin5_may            18
clim_6_tmin6_jun            18
clim_7_tmin7_jul            18
clim_8_tmin8_aug            18
clim_9_tmin9_sep            18
clim_10_tmin10_oct          18
clim_11_tmin11_nov          18
clim_12_tmin12_dec          18
clim_13_tmax1_jan           18
clim_14_tmax2_feb           18
clim_15_tmax3_mar           18
clim_16_tmax4_apr           18
clim_17_tmax5_may           18
clim_18_tmax6_jun           18
clim_19_tmax7_jul           18
clim_20_tmax8_aug           18
clim_21_tmax9_sep           18
clim_22_tmax10_oct          18
clim_23_tmax11_nov          18
clim_24_tmax12_dec          18
clim_25_prec1_jan           18
clim_26_prec2_feb           18
clim_27_prec3_mar           18
clim_28_prec4_apr           18
clim_29_prec5_may           18
clim_30_prec6_jun           18
clim_31_prec7_jul           18
clim_32_prec8_aug           18
clim_33_

**Amongst the numerical values**, we Removed the 11 clim_ columns with NaNs and removed remaining env_columns with NaNs

x Inserted median value in place of NaN for env_columns

In [55]:
# Identify columns that start with 'clim_' and 'env_'
clim_columns = [col for col in clim_data.columns if col.startswith('clim_')]
env_columns = [col for col in clim_data.columns if col.startswith('env_')]

# Remove rows where any 'clim_' column has NaN
coffee_data = coffee_data.dropna(subset=clim_columns)
#coffee_data = coffee_data.dropna(subset=env_columns)

missing_data = coffee_data.isnull().sum()
missing_data


clim_1_tmin1_jan             0
clim_2_tmin2_feb             0
clim_3_tmin3_mar             0
clim_4_tmin4_apr             0
clim_5_tmin5_may             0
clim_6_tmin6_jun             0
clim_7_tmin7_jul             0
clim_8_tmin8_aug             0
clim_9_tmin9_sep             0
clim_10_tmin10_oct           0
clim_11_tmin11_nov           0
clim_12_tmin12_dec           0
clim_13_tmax1_jan            0
clim_14_tmax2_feb            0
clim_15_tmax3_mar            0
clim_16_tmax4_apr            0
clim_17_tmax5_may            0
clim_18_tmax6_jun            0
clim_19_tmax7_jul            0
clim_20_tmax8_aug            0
clim_21_tmax9_sep            0
clim_22_tmax10_oct           0
clim_23_tmax11_nov           0
clim_24_tmax12_dec           0
clim_25_prec1_jan            0
clim_26_prec2_feb            0
clim_27_prec3_mar            0
clim_28_prec4_apr            0
clim_29_prec5_may            0
clim_30_prec6_jun            0
clim_31_prec7_jul            0
clim_32_prec8_aug            0
clim_33_

In [56]:
# Fill missing values in 'env_' columns with the median
# for col in env_columns:
#     median_value = coffee_data[col].median()
#     coffee_data[col].fillna(median_value, inplace=True)

medians = coffee_data[env_columns].median()

coffee_data.loc[:, env_columns] = coffee_data[env_columns].fillna(medians)

#coffee_data.shape
missing_data = coffee_data.isnull().sum()
missing_data

clim_1_tmin1_jan            0
clim_2_tmin2_feb            0
clim_3_tmin3_mar            0
clim_4_tmin4_apr            0
clim_5_tmin5_may            0
clim_6_tmin6_jun            0
clim_7_tmin7_jul            0
clim_8_tmin8_aug            0
clim_9_tmin9_sep            0
clim_10_tmin10_oct          0
clim_11_tmin11_nov          0
clim_12_tmin12_dec          0
clim_13_tmax1_jan           0
clim_14_tmax2_feb           0
clim_15_tmax3_mar           0
clim_16_tmax4_apr           0
clim_17_tmax5_may           0
clim_18_tmax6_jun           0
clim_19_tmax7_jul           0
clim_20_tmax8_aug           0
clim_21_tmax9_sep           0
clim_22_tmax10_oct          0
clim_23_tmax11_nov          0
clim_24_tmax12_dec          0
clim_25_prec1_jan           0
clim_26_prec2_feb           0
clim_27_prec3_mar           0
clim_28_prec4_apr           0
clim_29_prec5_may           0
clim_30_prec6_jun           0
clim_31_prec7_jul           0
clim_32_prec8_aug           0
clim_33_prec9_sep           0
clim_34_pr

In [57]:
coffee_data.shape

(450, 160)

In [58]:
# Identify constant columns (where min == max for each column)
constant_columns = clim_data.columns[clim_data.nunique() == 1].tolist()

# Print the constant columns
print(f"Constant columns: {constant_columns}")


Constant columns: []


In [59]:
env_data.shape

(468, 83)

In [60]:
missing_data = env_data.isnull().sum()
missing_data

env_75_geo_1_1    0
env_75_geo_10     0
env_75_geo_9      0
env_75_geo_4      0
env_75_geo_13     0
env_75_geo_7      0
env_75_geo_12     0
env_75_geo_6      0
env_75_geo_5      0
env_75_geo_11     0
env_75_geo_2      0
env_75_geo_0      0
env_76_soi_5      0
env_76_soi_9      0
env_76_soi_1      0
env_76_soi_7      0
env_76_soi_23     0
env_76_soi_18     0
env_76_soi_20     0
env_76_soi_11     0
env_76_soi_19     0
env_76_soi_10     0
env_76_soi_3      0
env_76_soi_2      0
env_76_soi_15     0
env_76_soi_12     0
env_76_soi_22     0
env_76_soi_21     0
env_76_soi_6      0
env_76_soi_16     0
env_76_soi_17     0
env_76_soi_8      0
env_76_soi_4      0
env_76_soi_14     0
env_76_soi_13     0
env_76_soi_0      0
env_77_veg_1      0
env_77_veg_2      0
env_77_veg_3      0
env_77_veg_4      0
env_77_veg_5      0
env_77_veg_6      0
env_77_veg_7      0
env_77_veg_9      0
env_77_veg_10     0
env_77_veg_11     0
env_77_veg_12     0
env_77_veg_13     0
env_77_veg_14     0
env_77_veg_15     0


Managing null columns in the categorial one-hot encoded data (meaning the unused categories)

In [61]:
print(env_data)

     env_75_geo_1_1  env_75_geo_10  env_75_geo_9  env_75_geo_4  env_75_geo_13  env_75_geo_7  env_75_geo_12  env_75_geo_6  env_75_geo_5  env_75_geo_11  ...  env_78_wat_25  env_78_wat_16  env_78_wat_17  env_78_wat_1.1  env_78_wat_18  env_78_wat_15  env_78_wat_19  env_78_wat_24  env_78_wat_21  env_78_wat_0
0                 0              1             0             0              0             0              0             0             0              0  ...              0              0              0               0              0              0              0              0              0             0
1                 0              1             0             0              0             0              0             0             0              0  ...              0              0              0               0              0              0              0              0              0             0
2                 0              1             0             0              0        

In [62]:
# Apply a lambda function to each column to check if the column is constant (i.e., all values are the same)
constant_columns = env_data.apply(lambda col: col.nunique() == 1, axis=0)

# Filter out the column names that are constant
constant_columns_list = constant_columns[constant_columns].index.tolist()
count = len(constant_columns_list)
# Print the constant columns
print(f"Constant columns: {count}")

num_columns = env_data.shape[1]

# Print the number of columns
print(f"Number of columns in the DataFrame: {num_columns}")



Constant columns: 14
Number of columns in the DataFrame: 83


In [63]:
for col in env_data.columns:
    print(f"{col}: {env_data[col].sum()}")

env_75_geo_1_1: 26
env_75_geo_10: 174
env_75_geo_9: 61
env_75_geo_4: 0
env_75_geo_13: 0
env_75_geo_7: 56
env_75_geo_12: 0
env_75_geo_6: 55
env_75_geo_5: 17
env_75_geo_11: 2
env_75_geo_2: 31
env_75_geo_0: 46
env_76_soi_5: 1
env_76_soi_9: 37
env_76_soi_1: 0
env_76_soi_7: 6
env_76_soi_23: 15
env_76_soi_18: 32
env_76_soi_20: 18
env_76_soi_11: 70
env_76_soi_19: 11
env_76_soi_10: 31
env_76_soi_3: 2
env_76_soi_2: 0
env_76_soi_15: 18
env_76_soi_12: 13
env_76_soi_22: 22
env_76_soi_21: 43
env_76_soi_6: 37
env_76_soi_16: 33
env_76_soi_17: 8
env_76_soi_8: 17
env_76_soi_4: 7
env_76_soi_14: 0
env_76_soi_13: 1
env_76_soi_0: 46
env_77_veg_1: 2
env_77_veg_2: 5
env_77_veg_3: 2
env_77_veg_4: 12
env_77_veg_5: 53
env_77_veg_6: 54
env_77_veg_7: 73
env_77_veg_9: 1
env_77_veg_10: 0
env_77_veg_11: 2
env_77_veg_12: 15
env_77_veg_13: 2
env_77_veg_14: 63
env_77_veg_15: 17
env_77_veg_16: 118
env_77_veg_18: 0
env_77_veg_19: 0
env_77_veg_22: 7
env_77_veg_23: 1
env_77_veg_25: 29
env_77_veg_0: 12
env_78_wat_9: 4
env_7

In [64]:
print(f"Shape of env_data: {env_data.shape}")

columns_to_remove = env_data.apply(lambda col: col.eq(0).all(), axis=0)

# Filter out the columns that are all 0
columns_to_remove = columns_to_remove[columns_to_remove].index.tolist()

# Print the columns that have all values as 0
print(f"Columns with all 0s: {columns_to_remove}")

Shape of env_data: (468, 83)
Columns with all 0s: ['env_75_geo_4', 'env_75_geo_13', 'env_75_geo_12', 'env_76_soi_1', 'env_76_soi_2', 'env_76_soi_14', 'env_77_veg_10', 'env_77_veg_18', 'env_77_veg_19', 'env_78_wat_7', 'env_78_wat_23', 'env_78_wat_25', 'env_78_wat_16', 'env_78_wat_21']


In [65]:

#Remove these columns from the DataFrame
env_data_clean = env_data.drop(columns=columns_to_remove)

# Print the columns that were removed
print(f"Removed columns: {columns_to_remove}")

# Check the shape of the DataFrame after removing the columns
print(f"Shape of env_data_clean: {env_data_clean.shape}")


Removed columns: ['env_75_geo_4', 'env_75_geo_13', 'env_75_geo_12', 'env_76_soi_1', 'env_76_soi_2', 'env_76_soi_14', 'env_77_veg_10', 'env_77_veg_18', 'env_77_veg_19', 'env_78_wat_7', 'env_78_wat_23', 'env_78_wat_25', 'env_78_wat_16', 'env_78_wat_21']
Shape of env_data_clean: (468, 69)


In [66]:
# Remove these columns from the DataFrame
coffee_data_cleaned = coffee_data.drop(columns=columns_to_remove)

# Print the columns that were removed
print(f"Removed columns: {columns_to_remove}")

# Check the shape of the DataFrame after removing the columns
print(f"Shape of coffee_data: {coffee_data_cleaned.shape}")

Removed columns: ['env_75_geo_4', 'env_75_geo_13', 'env_75_geo_12', 'env_76_soi_1', 'env_76_soi_2', 'env_76_soi_14', 'env_77_veg_10', 'env_77_veg_18', 'env_77_veg_19', 'env_78_wat_7', 'env_78_wat_23', 'env_78_wat_25', 'env_78_wat_16', 'env_78_wat_21']
Shape of coffee_data: (450, 146)


Now that all the NaN and constant columns have been managed or removed We are looking into the outliers using <span style="color:teal">env_data_clean</span>
 and <span style="color:teal">clim_data</span> :

 ### After analysis no outliers will be removed

In [67]:
coffee_data_cleaned.to_csv(r"../data/cleaned_data_w_class.csv", index=False)
coffee_data_cleaned.head()

,clim_1_tmin1_jan,clim_2_tmin2_feb,clim_3_tmin3_mar,clim_4_tmin4_apr,clim_5_tmin5_may,clim_6_tmin6_jun,clim_7_tmin7_jul,clim_8_tmin8_aug,clim_9_tmin9_sep,clim_10_tmin10_oct,...,env_78_wat_22,env_78_wat_17,env_78_wat_1.1,env_78_wat_18,env_78_wat_15,env_78_wat_19,env_78_wat_24,env_78_wat_0,caffeine_percent,caffeine_class
0,195.5,195.0,186.0,169.0,124.0,108.0,108.0,119.0,133.0,173.0,...,0,0,0,0,0,0,0,0,0.0,0
1,177.5,177.0,170.0,153.0,121.0,108.0,107.0,111.0,122.0,157.0,...,0,0,0,0,0,0,0,0,0.0,0
2,177.5,177.0,170.0,153.0,121.0,108.0,107.0,111.0,122.0,157.0,...,0,0,0,0,0,0,0,0,0.0,0
3,208.5,208.0,200.0,182.0,146.0,132.0,128.0,133.0,149.0,185.0,...,0,0,0,0,0,0,0,0,0.0,0
4,177.5,177.0,170.0,153.0,121.0,108.0,107.0,111.0,122.0,157.0,...,0,0,0,0,0,0,0,0,0.0,0
